# Input pool

Builds the input pool the search picks from, for three backbones: CLIP ViT-B/16, CLIP ViT-L/14 and EfficientNet-B4.

- **128 PCA components**, fitted on all of FF++ train (no labels).
- **16 PLS components**, fitted on search-train only, so the inner fitness frames never touch a label.
- Every column is mapped to an angle in [0, pi] using the 0.5 and 99.5 percentiles on train.


In [1]:
backbones = {
    "b16": ("features", "_clip.npy"),
    "l14": ("features_l14", "_clip.npy"),
    "effb4": ("features", "_effb4.npy"),
}
names = ["ffpp_train", "ffpp_val", "ffpp_test", "celebdf_test"]
n_pca = 128
n_pls = 16
low, high = 0.5, 99.5
check_celebdf = False
out_dir = "/kaggle/working/pool"
seed = 0

In [2]:
import os
import glob
import json

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

os.makedirs(out_dir, exist_ok=True)

## Load

All backbones were extracted from the same manifests, so their rows must match line by line. The check below stops the notebook if they don't.
B/16 and L/14 both save `*_clip.npy`, so each file is looked up by the name of the folder it sits in.

In [3]:
def find_all(filename):
    return sorted(glob.glob("/kaggle/input/**/" + filename, recursive=True))


def find_in(folder, filename):
    hits = [p for p in find_all(filename) if os.path.basename(os.path.dirname(p)) == folder]
    assert len(hits) == 1, (folder, filename, hits)
    return hits[0]


feats = {b: {} for b in backbones}
rows = {}
for b, (folder, suffix) in backbones.items():
    for name in names:
        path = find_in(folder, name + suffix)
        feats[b][name] = np.load(path).astype(np.float64)
        r = pd.read_csv(os.path.join(os.path.dirname(path), name + "_rows.csv"), dtype={"video_id": str})
        assert len(r) == len(feats[b][name]), (b, name)
        if name in rows:
            assert (r["path"].values == rows[name]["path"].values).all(), ("rows differ", b, name)
        else:
            rows[name] = r
        print(b, name, feats[b][name].shape)

# every attached copy of the inner split must be the same
copies = [np.load(p).astype(bool) for p in find_all("ffpp_train_is_inner.npy")]
assert len(copies) > 0
for c in copies:
    assert (c == copies[0]).all()
is_inner = copies[0]
assert len(is_inner) == len(rows["ffpp_train"])
search = ~is_inner
y = {name: rows[name]["label"].to_numpy() for name in names}
print("search-train frames", search.sum(), "| inner frames", is_inner.sum(), "| copies of the split", len(copies))

b16 ffpp_train (46076, 512)
b16 ffpp_val (22396, 512)
b16 ffpp_test (22400, 512)
b16 celebdf_test (16572, 512)
l14 ffpp_train (46076, 768)
l14 ffpp_val (22396, 768)
l14 ffpp_test (22400, 768)
l14 celebdf_test (16572, 768)
effb4 ffpp_train (46076, 1792)
effb4 ffpp_val (22396, 1792)
effb4 ffpp_test (22400, 1792)
effb4 celebdf_test (16572, 1792)
search-train frames 39164 | inner frames 6912 | copies of the split 1


## PCA, PLS and angles

Each feature vector is scaled to length 1 first, as in the earlier notebooks. PCA sees all of train; PLS sees only search-train frames and their labels.

In [4]:
pool = {b: {} for b in backbones}
scores = {b: {} for b in backbones}
columns = [f"pca{j}" for j in range(n_pca)] + [f"pls{j}" for j in range(n_pls)]

for b in backbones:
    xn = {name: feats[b][name] / np.linalg.norm(feats[b][name], axis=1, keepdims=True) for name in names}

    pca = PCA(n_components=n_pca, svd_solver="full", random_state=seed)
    pca.fit(xn["ffpp_train"])
    pls = PLSRegression(n_components=n_pls, scale=True)
    pls.fit(xn["ffpp_train"][search], y["ffpp_train"][search])

    for name in names:
        scores[b][name] = np.hstack([pca.transform(xn[name]), pls.transform(xn[name])])
    lo = np.percentile(scores[b]["ffpp_train"], low, axis=0)
    hi = np.percentile(scores[b]["ffpp_train"], high, axis=0)
    for name in names:
        pool[b][name] = np.clip((scores[b][name] - lo) / (hi - lo), 0.0, 1.0) * np.pi
        np.save(f"{out_dir}/{b}_{name}_pool.npy", pool[b][name].astype(np.float32))
        np.save(f"{out_dir}/{b}_{name}_scores.npy", scores[b][name].astype(np.float32))

    np.savez(f"{out_dir}/{b}_pool_params.npz", pca_components=pca.components_, pca_mean=pca.mean_,
             pca_variance_ratio=pca.explained_variance_ratio_, pls_rotations=pls.x_rotations_,
             pls_x_mean=pls._x_mean, pls_x_std=pls._x_std, angle_lo=lo, angle_hi=hi)
    cum = np.cumsum(pca.explained_variance_ratio_)
    print(b, "variance kept by 8, 32, 128 PCA components: %.1f%%, %.1f%%, %.1f%%"
          % (100 * cum[7], 100 * cum[31], 100 * cum[127]))

for name in names:
    rows[name].to_csv(f"{out_dir}/{name}_rows.csv", index=False)
np.save(f"{out_dir}/ffpp_train_is_inner.npy", is_inner)
with open(f"{out_dir}/columns.json", "w") as f:
    json.dump(columns, f)

b16 variance kept by 8, 32, 128 PCA components: 42.5%, 69.4%, 92.6%
l14 variance kept by 8, 32, 128 PCA components: 35.4%, 60.3%, 85.7%
effb4 variance kept by 8, 32, 128 PCA components: 48.8%, 70.4%, 87.7%


## Per-column strength

AUC of each pool column on its own, on search-train and on val. A column can point either way, so strength is max(auc, 1 - auc). The greedy and mutual-information baselines will use the search-train numbers only.

In [5]:
strength = {}
for b in backbones:
    table = []
    for j, col in enumerate(columns):
        a_st = roc_auc_score(y["ffpp_train"][search], pool[b]["ffpp_train"][search, j])
        a_val = roc_auc_score(y["ffpp_val"], pool[b]["ffpp_val"][:, j])
        table.append({"column": col, "strength_search_train": max(a_st, 1 - a_st), "strength_val": max(a_val, 1 - a_val)})
    strength[b] = pd.DataFrame(table)
    strength[b].to_csv(f"{out_dir}/{b}_column_strength.csv", index=False)
    top = strength[b].sort_values("strength_search_train", ascending=False).head(8)
    print(b, "strongest 8 on search-train:", top["column"].tolist())

b16 strongest 8 on search-train: ['pls0', 'pls1', 'pca8', 'pls3', 'pca17', 'pls2', 'pca3', 'pls5']
l14 strongest 8 on search-train: ['pls0', 'pca1', 'pls1', 'pca5', 'pca8', 'pca2', 'pls4', 'pca11']
effb4 strongest 8 on search-train: ['pls4', 'pls6', 'pls7', 'pls0', 'pls1', 'pls2', 'pls3', 'pls5']


## Bottleneck table

Logistic regression on standardized inputs, scored at video level (a video's score is the mean of its frame probabilities). For every row, C is picked from 0.001, 0.01, 0.1 and 1 on the inner fitness frames (trained on search-train), then the model is refit on all the training frames it is allowed to use.

Two kinds of rows:
- **raw**: the PCA or PLS scores themselves.
- **twin**: [sin, cos] of the 8 angles. This is exactly the function class of the additive circuit (RY encoding, single-qubit rotations, weighted sum of Z), so it shows what that circuit can reach with those 8 inputs.

Three numbers per row:
- **val**: trained on all of FF++ train, scored on FF++ val.
- **held-out mean** and **held-out worst**: leave one manipulation out. Train without that manipulation's fakes, score on val real videos plus that manipulation's fakes only. PLS, the angle range and the "best 8" ranking are refit inside each fold without that manipulation, so nothing that picks or builds the inputs has seen it. PCA uses no labels and stays fixed.
- **celebdf**: only when `check_celebdf` is on.

In [6]:
fake_methods = sorted(rows["ffpp_train"].loc[y["ffpp_train"] == 1, "method"].unique())
video_ids = {name: rows[name].groupby(["method", "video_id"]).ngroup().to_numpy() for name in names}
train_method = rows["ffpp_train"]["method"].to_numpy()
val_method = rows["ffpp_val"]["method"].to_numpy()
c_grid = [0.001, 0.01, 0.1, 1.0]
print("held-out methods:", fake_methods)

eval_names = ["ffpp_train", "ffpp_val"]
if check_celebdf:
    eval_names.append("celebdf_test")


def video_auc(name, prob, keep):
    df = pd.DataFrame({"video": video_ids[name][keep], "p": prob[keep], "y": y[name][keep]})
    g = df.groupby("video").agg(p=("p", "mean"), y=("y", "first"))
    return roc_auc_score(g["y"], g["p"])


def fit(x, keep, c):
    model = make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=5000))
    model.fit(x[keep], y["ffpp_train"][keep])
    return model


def fit_tuned(x, keep):
    # pick C on the inner frames, then refit on every kept train frame
    best_c, best_auc = None, -1.0
    for c in c_grid:
        model = fit(x, keep & search, c)
        auc = video_auc("ffpp_train", model.predict_proba(x)[:, 1], keep & is_inner)
        if auc > best_auc:
            best_c, best_auc = c, auc
    return fit(x, keep, best_c), best_c


def twin(a):
    return np.hstack([np.sin(a), np.cos(a)])


def make_inputs(b, keep):
    # PCA has no labels and stays fixed; PLS, the angle range and the ranking only see kept search-train frames
    xn = {n: feats[b][n] / np.linalg.norm(feats[b][n], axis=1, keepdims=True) for n in eval_names}
    fit_rows = search & keep
    pls = PLSRegression(n_components=n_pls, scale=True)
    pls.fit(xn["ffpp_train"][fit_rows], y["ffpp_train"][fit_rows])
    s = {n: np.hstack([scores[b][n][:, :n_pca], pls.transform(xn[n])]) for n in eval_names}
    lo = np.percentile(s["ffpp_train"][keep], low, axis=0)
    hi = np.percentile(s["ffpp_train"][keep], high, axis=0)
    a = {n: np.clip((s[n] - lo) / (hi - lo), 0.0, 1.0) * np.pi for n in eval_names}
    col_strength = []
    for j in range(n_pca + n_pls):
        auc = roc_auc_score(y["ffpp_train"][fit_rows], a["ffpp_train"][fit_rows, j])
        col_strength.append(max(auc, 1 - auc))
    best8 = np.argsort(col_strength)[::-1][:8].tolist()
    first8 = list(range(8))
    pls8 = list(range(n_pca, n_pca + 8))
    pls16 = list(range(n_pca, n_pca + n_pls))
    return [
        ("raw first 8 PCA", {n: s[n][:, first8] for n in eval_names}),
        ("raw best 8 of pool", {n: s[n][:, best8] for n in eval_names}),
        ("raw PLS 8", {n: s[n][:, pls8] for n in eval_names}),
        ("raw PLS 16", {n: s[n][:, pls16] for n in eval_names}),
        ("raw all 144", {n: s[n] for n in eval_names}),
        ("twin first 8 PCA", {n: twin(a[n][:, first8]) for n in eval_names}),
        ("twin best 8 of pool", {n: twin(a[n][:, best8]) for n in eval_names}),
        ("twin PLS 8", {n: twin(a[n][:, pls8]) for n in eval_names}),
        ("full linear probe", xn),
    ]


results = []
all_train = np.ones(len(y["ffpp_train"]), dtype=bool)
all_val = np.ones(len(y["ffpp_val"]), dtype=bool)
for b in backbones:
    folds = [("all", all_train)] + [(m, (y["ffpp_train"] == 0) | (train_method != m)) for m in fake_methods]
    table = {}
    for fold, keep in folds:
        for label, x in make_inputs(b, keep):
            row = table.setdefault(label, {"backbone": b, "input": label})
            model, c = fit_tuned(x["ffpp_train"], keep)
            if fold == "all":
                row["C"] = c
                row["val"] = video_auc("ffpp_val", model.predict_proba(x["ffpp_val"])[:, 1], all_val)
                if check_celebdf:
                    celeb_all = np.ones(len(y["celebdf_test"]), dtype=bool)
                    row["celebdf"] = video_auc("celebdf_test", model.predict_proba(x["celebdf_test"])[:, 1], celeb_all)
            else:
                val_keep = (y["ffpp_val"] == 0) | (val_method == fold)
                row["held-out " + fold] = video_auc("ffpp_val", model.predict_proba(x["ffpp_val"])[:, 1], val_keep)
        print(b, fold, "done")
    for row in table.values():
        held = [row["held-out " + m] for m in fake_methods]
        row["held-out mean"] = np.mean(held)
        row["held-out worst"] = np.min(held)
        results.append(row)

results = pd.DataFrame(results)
results.to_csv(f"{out_dir}/bottleneck_table.csv", index=False)

order = list(table)
shown = ["val", "held-out mean", "held-out worst"]
if check_celebdf:
    shown.append("celebdf")
for metric in shown + ["C"]:
    print()
    print(metric)
    print(results.pivot(index="input", columns="backbone", values=metric).loc[order, list(backbones)].round(3).to_string())

held-out methods: ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']
b16 all done
b16 Deepfakes done
b16 Face2Face done
b16 FaceSwap done
b16 NeuralTextures done
l14 all done
l14 Deepfakes done
l14 Face2Face done
l14 FaceSwap done
l14 NeuralTextures done
effb4 all done
effb4 Deepfakes done
effb4 Face2Face done
effb4 FaceSwap done
effb4 NeuralTextures done

val
backbone               b16    l14  effb4
input                                   
raw first 8 PCA      0.636  0.739  0.522
raw best 8 of pool   0.828  0.855  0.692
raw PLS 8            0.850  0.891  0.692
raw PLS 16           0.873  0.896  0.720
raw all 144          0.877  0.898  0.725
twin first 8 PCA     0.645  0.738  0.520
twin best 8 of pool  0.832  0.851  0.689
twin PLS 8           0.852  0.890  0.689
full linear probe    0.890  0.905  0.756

held-out mean
backbone               b16    l14  effb4
input                                   
raw first 8 PCA      0.630  0.738  0.508
raw best 8 of pool   0.777  0.798  0.574
r

The held-out AUC for each manipulation, per backbone:

In [7]:
held_cols = ["held-out " + m for m in fake_methods]
for b in backbones:
    print()
    print(b)
    print(results[results["backbone"] == b].set_index("input")[held_cols].round(3).to_string())


b16
                     held-out Deepfakes  held-out Face2Face  held-out FaceSwap  held-out NeuralTextures
input                                                                                                  
raw first 8 PCA                   0.806               0.562              0.680                    0.474
raw best 8 of pool                0.932               0.715              0.896                    0.564
raw PLS 8                         0.926               0.728              0.865                    0.589
raw PLS 16                        0.932               0.776              0.868                    0.614
raw all 144                       0.937               0.787              0.870                    0.616
twin first 8 PCA                  0.834               0.553              0.680                    0.480
twin best 8 of pool               0.936               0.716              0.895                    0.565
twin PLS 8                        0.926               0.729

## Saved files

`{backbone}_{split}_pool.npy` (angles, 144 columns), `{backbone}_{split}_scores.npy` (the raw scores), `{backbone}_pool_params.npz`, `{backbone}_column_strength.csv`, `columns.json`, the rows files, `ffpp_train_is_inner.npy` and `bottleneck_table.csv`. Save this output as a dataset for the next notebooks.

In [8]:
!ls -lh {out_dir}

total 368M
-rw-r--r-- 1 root root 9.2M Nov 26 13:15 b16_celebdf_test_pool.npy
-rw-r--r-- 1 root root 9.2M Nov 26 13:15 b16_celebdf_test_scores.npy
-rw-r--r-- 1 root root 6.2K Nov 26 13:16 b16_column_strength.csv
-rw-r--r-- 1 root root  13M Nov 26 13:15 b16_ffpp_test_pool.npy
-rw-r--r-- 1 root root  13M Nov 26 13:15 b16_ffpp_test_scores.npy
-rw-r--r-- 1 root root  26M Nov 26 13:15 b16_ffpp_train_pool.npy
-rw-r--r-- 1 root root  26M Nov 26 13:15 b16_ffpp_train_scores.npy
-rw-r--r-- 1 root root  13M Nov 26 13:15 b16_ffpp_val_pool.npy
-rw-r--r-- 1 root root  13M Nov 26 13:15 b16_ffpp_val_scores.npy
-rw-r--r-- 1 root root 594K Nov 26 13:15 b16_pool_params.npz
-rw-r--r-- 1 root root 4.3K Nov 26 13:34 bottleneck_table.csv
-rw-r--r-- 1 root root 2.3M Nov 26 13:16 celebdf_test_rows.csv
-rw-r--r-- 1 root root 1.3K Nov 26 13:16 columns.json
-rw-r--r-- 1 root root 9.2M Nov 26 13:16 effb4_celebdf_test_pool.npy
-rw-r--r-- 1 root root 9.2M Nov 26 13:16 effb4_celebdf_test_scores.npy
-rw-r--r-- 1 root 